# T03 · Training on the full 17×17 board

We train on the full 17×17 board with the hyperparameters that T02 found on the 9×9 board: the same three
curriculum stages, and for each network its own best learning rates and entropy coefficient. This only works
once the crate reward is doubled (`CRATE_REWARD`). On the larger board the next coin is further away from a
destroyed crate, so bombing needs a larger reward to pay off.

The notebook saves everything the analysis notebooks need: the training metrics of every stage (including the
number of bombs planted), the weights after every stage, and the result of every evaluation game.

**Writes** `runs/a2c_17x17/` (or `runs/a2c_17x17_03/` with `CRATE_REWARD = 0.3`): `history.csv`, `eval.csv`
(one row per evaluation game), `config.json` and `<network>/<stage>.eqx`. **Used by** A05, A06 and T04.

In [ ]:
import os

os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", os.path.expanduser("~/.cache/jax"))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.16")

import dataclasses
import json
import pathlib
import time

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

from bomberman.agents import A2CObjective, PolicyAgent, RuleBasedAgent
from bomberman.agents.networks import KartalCNNActorCriticNetwork, SkynetCNNActorCriticNetwork
from bomberman.game import ACTION, Game
from bomberman.training import (
    CoinPickupReward,
    DeathPenalty,
    KillReward,
    PBRSReward,
    PrintLogger,
    Rollout,
    StepPenalty,
    Training,
    UrgentCrateReward,
    UrgentStepPenalty,
    coin_task_settings,
    crate_task_settings,
    versus_task_settings,
)
from bomberman.training.scenarios import NoReward

HERE = pathlib.Path.cwd().resolve()  # run from the notebook's folder or from the repository root
EXP = next(p / "experiments" for p in (HERE, *HERE.parents) if (p / "experiments" / "README.md").exists())
RUNS = EXP / "runs"
print(f"backend: {jax.default_backend()}")

## Configuration

Each network uses the best settings of its own search in T02 (`runs/a2c_hpo*/best.json`).

In [ ]:
SIZE = 17
NETWORKS = {"FC128": KartalCNNActorCriticNetwork, "Conv1x1": SkynetCNNActorCriticNetwork}
HPO_RUNS = {"FC128": "a2c_hpo", "Conv1x1": "a2c_hpo_1x1"}
TRAIN = ("FC128",)  # networks to train; add "Conv1x1" for the Skynet network
HPO = {name: json.loads((EXP / "runs" / HPO_RUNS[name] / "best.json").read_text()) for name in TRAIN}
ENTROPY_FINAL = 5e-4  # as in the search
CRATE_REWARD = 0.6  # twice the 0.3 used on 9x9; with 0.3 you get the failed run from the report
OUT = RUNS / ("a2c_17x17" if CRATE_REWARD == 0.6 else f"a2c_17x17_{CRATE_REWARD:g}".replace("0.", "0"))
OUT.mkdir(parents=True, exist_ok=True)
print(f"output: {OUT}")

BUDGET = dict(num_envs=256, iterations=(2000, 2000, 1500), eval_episodes=128, log_every=50)
SEED = 1

EVAL_SETTINGS = versus_task_settings(size=SIZE, crate_density=0.35, coins=9, max_steps=200, agents=2)
pd.DataFrame(HPO).T

## Curriculum

One function per stage, as in T02, but on the 17×17 board and with longer games. Each one continues training
the agent from the previous stage with a new entropy schedule and returns `(agent, history, info)`.

In [ ]:
def train_stage(agent, key, settings, reward, *, iterations, num_envs, entropy, learning_rate=3e-4,
                forbid_bomb=False, opponents=()):
    c0, c1 = entropy  # c_H decays linearly from c0 to c1 during the stage
    agent = agent.configure(objective=agent.objective.configure(entropy_coef=c0, entropy_final=c1))
    trainer = Training(settings, [agent, *opponents], reward_fn=reward, forbid_bomb=forbid_bomb,
                       num_envs=num_envs, learning_rate=learning_rate,
                       logger=PrintLogger(every=BUDGET["log_every"], keys=("agent0/coins", "agent0/death",
                                                                             "agent0/bombs", "agent0/entropy")))
    agent, *_ = trainer.fit(key, iterations)
    info = dict(iterations=iterations, entropy=entropy, learning_rate=learning_rate, forbid_bomb=forbid_bomb,
                settings=settings, reward=reward, opponents=[type(o).__name__ for o in opponents])
    return agent, trainer.history, info


def collect(agent, key, **kwargs):
    """Stage 1, collect coins: no crates, 8 coins, no bombs allowed."""
    settings = coin_task_settings(size=SIZE, coins=8, max_steps=200)
    reward = CoinPickupReward() + StepPenalty(step=0.02)
    return train_stage(agent, key, settings, reward, forbid_bomb=True, **kwargs)


def crates(agent, key, **kwargs):
    """Stage 2, bomb crates: crate density 0.18, 9 coins, most of them under crates, doubled crate reward."""
    settings = crate_task_settings(size=SIZE, coins=9, crate_density=0.18, max_steps=200)
    reward = CoinPickupReward() + UrgentCrateReward(crate=CRATE_REWARD) + DeathPenalty(death=2.0) + UrgentStepPenalty()
    return train_stage(agent, key, settings, reward, **kwargs)


def versus(agent, key, *, pbrs=True, **kwargs):
    """Stage 3, 1 vs. 1 against the rule-based agent, with potential-based reward shaping."""
    settings = versus_task_settings(size=SIZE, coins=6, crate_density=0.18, max_steps=200, agents=2)
    reward = (CoinPickupReward() + KillReward() + DeathPenalty(death=3.0) + StepPenalty(step=0.02)
              + UrgentCrateReward(urgent_crate_scale=1.0) + PBRSReward(settings, enabled=pbrs))
    return train_stage(agent, key, settings, reward, opponents=(RuleBasedAgent(),), **kwargs)


CURRICULUM = (collect, crates, versus)

## Agent, training loop and evaluation

`TrackedA2C` is A2C that also logs the number of bombs planted per game.

In [ ]:
class TrackedA2C(A2CObjective):
    def __call__(self, network, batch):
        loss, metrics = super().__call__(network, batch)
        bombs = ((batch.action == ACTION.BOMB) & batch.mask).sum(-1).mean()
        return loss, {**metrics, "bombs": bombs}


def history_frame(history, **labels):
    df = pd.DataFrame({k: np.asarray(v) for k, v in history.items()})
    df = df.rename(columns=lambda c: c.replace("agent0/", "").replace("agent1/", "opp_"))
    return df.assign(stage_iteration=np.arange(len(df)), **labels)


def train_curriculum(name, key):
    """Train through all stages with the best settings T02 found for network `name`.

    Returns (agent, history, {stage: network}, {stage: info}).
    """
    hpo = HPO[name]
    init_key, key = jax.random.split(key)
    agent = PolicyAgent(NETWORKS[name].for_settings(EVAL_SETTINGS, init_key),
                        TrackedA2C(gamma=0.99, entropy_coef=hpo["entropy_coef"], entropy_final=ENTROPY_FINAL,
                                   value_coef=0.5, n_steps=5))
    frames, networks, infos, offset = [], {}, {}, 0
    keys = jax.random.split(key, len(CURRICULUM))
    for index, (stage, iterations, stage_key) in enumerate(zip(CURRICULUM, BUDGET["iterations"], keys)):
        stage_name = stage.__name__
        agent, history, infos[stage_name] = stage(
            agent, stage_key, iterations=iterations, num_envs=BUDGET["num_envs"],
            learning_rate=hpo[f"learning_rate_{stage_name}"], entropy=(hpo["entropy_coef"], ENTROPY_FINAL))
        frame = history_frame(history, network=name, stage=stage_name, stage_index=index)
        frames.append(frame.assign(iteration=frame.stage_iteration + offset))
        networks[stage_name] = agent.network
        offset += iterations
    return agent, pd.concat(frames, ignore_index=True), networks, infos


EVAL_ROLLOUT = Rollout(Game(settings=EVAL_SETTINGS), NoReward(), EVAL_SETTINGS.rules.max_steps)


def seat_stats(out, seat=0):
    """The result of every game for one player. A self-kill is a death in the player's own explosion."""
    states = out.states
    after = jax.tree.map(lambda a: a[:, 1:], states)
    died = states.alive[:, :-1, seat] & ~states.alive[:, 1:, seat]
    hit = jax.vmap(jax.vmap(lambda board: board.explosions.covers(board.agent_positions)))(after.board)
    score = states.score[:, -1]
    rival = jnp.delete(score, seat, axis=1, assume_unique_indices=True).max(axis=1)
    return dict(
        score=score[:, seat], rival_score=rival, win=score[:, seat] > rival, coins=out.coins[:, seat],
        death=~out.alive[:, seat], self_kill=(died & hit[:, :, seat, seat]).any(axis=1), steps=out.steps,
    )


@eqx.filter_jit
def evaluate(network, key, episodes):
    agent = PolicyAgent(network, A2CObjective())
    return seat_stats(EVAL_ROLLOUT([agent, RuleBasedAgent()], key, greedy=True, num_envs=episodes))

## Run

In [ ]:
histories, games, infos_by_network = [], [], {}
for name in TRAIN:
    start = time.time()
    train_key, eval_key = jax.random.split(jax.random.key(SEED))
    agent, history, networks, infos_by_network[name] = train_curriculum(name, train_key)
    (OUT / name).mkdir(exist_ok=True)
    for stage, network in networks.items():
        eqx.tree_serialise_leaves(OUT / name / f"{stage}.eqx", network)

    stats = {k: np.asarray(v, float) for k, v in evaluate(agent.network, eval_key, BUDGET["eval_episodes"]).items()}
    games += [dict(network=name, episode=i, **{k: v[i] for k, v in stats.items()}) for i in range(len(stats["score"]))]
    histories.append(history)
    print(f"{name}: score {stats['score'].mean():.2f}  win {stats['win'].mean():.2f}  "
          f"self-kill {stats['self_kill'].mean():.2f}  ({(time.time() - start) / 60:.0f} min)")

In [ ]:
def fields(module, skip=("settings",)):
    return {f.name: getattr(module, f.name) for f in dataclasses.fields(module) if f.name not in skip}


def terms(reward):
    """A combined reward as a list of its parts, e.g. [{"DeathPenalty": {"death": 3.0}}, ...], for config.json."""
    if hasattr(reward, "left"):
        return terms(reward.left) + terms(reward.right)
    return [{type(reward).__name__: fields(reward)}]


pd.concat(histories, ignore_index=True).to_csv(OUT / "history.csv", index=False)
evaluation = pd.DataFrame(games)
evaluation.to_csv(OUT / "eval.csv", index=False)
infos = next(iter(infos_by_network.values()))
(OUT / "config.json").write_text(json.dumps(dict(
    size=SIZE, seed=SEED, budget=BUDGET, networks={n: NETWORKS[n].__name__ for n in TRAIN}, hpo=HPO,
    entropy_final=ENTROPY_FINAL, crate_reward=CRATE_REWARD,
    stages=[dict(name=name, iterations=i["iterations"], forbid_bomb=i["forbid_bomb"], entropy=i["entropy"],
                 learning_rate=i["learning_rate"],
                 scenario=fields(i["settings"].scenario), rules=fields(i["settings"].rules),
                 reward=terms(i["reward"]), opponents=i["opponents"]) for name, i in infos.items()],
    eval=dict(scenario=fields(EVAL_SETTINGS.scenario), rules=fields(EVAL_SETTINGS.rules),
              episodes=BUDGET["eval_episodes"], opponent="RuleBasedAgent"),
), indent=2, default=str))
evaluation.groupby("network")[["score", "win", "coins", "death", "self_kill"]].mean().round(3)